# 04 — Chapter B data preparation (One-Class Novelty Detection)

**Project:** Network Traffic Clustering and Novelty Detection
**Reference:** *Project Proposal V2*, §5 (Chapter B), Phase 3 and Phase 7.

**Purpose.** Build the data for the one-class model of *normal* traffic, honouring
the proposal's scientific constraints. **No model is trained here.**

**The normal-traffic definition is `label == 0`.** The preparation:

1. Take the normal flows of the official training split (`label == 0`).
2. Split them into `normal_train` / `normal_val` with a fixed seed, **stratified by
   `service`**.
3. Keep the official **testing split** as the held-out *mixture* of normal + attack
   flows — untouched during fitting.
4. Fit all preprocessing **only on `normal_train`**; transform `normal_val` and the
   test set with the identical fitted transformers.
5. Keep all reference labels **separate** from the feature matrices.

**Why stratify by `service` (and not `proto`)?** `normal_val` is used later to pick
the flagging threshold, so it must reflect the *same mix of normal behaviours* as
`normal_train`. `service` (13 categories) is the right granularity: it is
meaningful and every category has ≥ 2 members. `proto` has 131 categories with many
singletons, which makes stratified splitting unstable — so we do not use it.

**Leakage rules enforced:** no attack flow ever reaches `normal_train`/`normal_val`;
no test row is used to fit any transformer; no test label is inspected to make a
preprocessing decision.

## 0. Environment setup

In [1]:
import sys, joblib
from pathlib import Path

def _find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook inside the project repository.")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from config.config import (
    CHAPTER_B_DIR, REPORTS_DIR, METRICS_DIR,
    RANDOM_SEED, ID_COLUMNS, LABEL_COLUMN, ATTACK_CAT_COLUMN,
)
from src.data.preprocess import (
    load_official_datasets, split_features_labels, check_finite, FeaturePipeline,
)

RANDOM_STATE = RANDOM_SEED
np.random.seed(RANDOM_STATE)
CHAPTER_B_DIR.mkdir(parents=True, exist_ok=True)
print("Chapter B output dir:", CHAPTER_B_DIR)

Chapter B output dir: /home/aminelby/Network-Traffic-Clustering/data/processed/chapter_b


## 1. Load and isolate normal traffic

**Purpose.** The model of normal is built from `label == 0` flows only. This is the
*one* place `label` is used as a filter (Proposal V2 §3, §6); it is never a feature.

In [2]:
train_df, test_df = load_official_datasets()

normal_df = train_df[train_df[LABEL_COLUMN] == 0].reset_index(drop=True)
assert (normal_df[LABEL_COLUMN] == 0).all()
assert (normal_df[ATTACK_CAT_COLUMN] == "Normal").all()
print("Normal flows in training split:", len(normal_df))
print("Attack flows in training split (held aside, NOT used for the normal model):",
      int((train_df[LABEL_COLUMN] == 1).sum()))

Normal flows in training split: 37000
Attack flows in training split (held aside, NOT used for the normal model): 45332


## 2. Split normal traffic into `normal_train` / `normal_val`

**Purpose.** 80/20 seeded, **stratified by `service`**, so `normal_val` carries the
same service mix as `normal_train`. The two sets are disjoint and their union is all
normal flows.

In [3]:
strat_key = normal_df["service"].astype(str)
try:
    idx_tr, idx_va = train_test_split(
        normal_df.index.to_numpy(),
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=strat_key.to_numpy(),
    )
    stratify_used = "service"
except ValueError as exc:  # safety net if a category is too small
    idx_tr, idx_va = train_test_split(
        normal_df.index.to_numpy(), test_size=0.20, random_state=RANDOM_STATE
    )
    stratify_used = f"none (fallback: {exc})"

normal_train_df = normal_df.loc[idx_tr].reset_index(drop=True)
normal_val_df = normal_df.loc[idx_va].reset_index(drop=True)

print("stratification used:", stratify_used)
print("normal_train:", normal_train_df.shape, "| normal_val:", normal_val_df.shape)
print("disjoint:", set(idx_tr).isdisjoint(set(idx_va)),
      "| union size:", len(set(idx_tr) | set(idx_va)), "== normal total:",
      len(set(idx_tr) | set(idx_va)) == len(normal_df))

# Distribution sanity-check across the two normal splits.
dist = pd.DataFrame({
    "normal_train": normal_train_df["service"].value_counts(normalize=True),
    "normal_val": normal_val_df["service"].value_counts(normalize=True),
}).fillna(0).round(4)
display(dist)
print("max service-share gap between splits:",
      float((dist["normal_train"] - dist["normal_val"]).abs().max()))

stratification used: service
normal_train: (29600, 45) | normal_val: (7400, 45)
disjoint: True | union size: 37000 == normal total: True


,normal_train,normal_val
service,,
-,0.7399,0.7399
dns,0.0829,0.0828
ftp,0.0205,0.0205
ftp-data,0.0256,0.0257
http,0.1084,0.1085
radius,0.0001,0.0000
smtp,0.0172,0.0172
ssh,0.0054,0.0054


max service-share gap between splits: 0.00010000000000000286


## 3. Define the held-out test mixture

**Purpose.** The official testing split (normal **and** attack) is the one-shot
evaluation set of Phase 8. It is loaded, split into features/labels, and otherwise
left untouched — no fitting happens on it.

In [4]:
X_train_norm_raw, y_normal_train = split_features_labels(normal_train_df)
X_val_norm_raw,   y_normal_val   = split_features_labels(normal_val_df)
X_test_raw,       y_test         = split_features_labels(test_df)

forbidden = set(ID_COLUMNS + [LABEL_COLUMN, ATTACK_CAT_COLUMN])
for frame in (X_train_norm_raw, X_val_norm_raw, X_test_raw):
    assert not (forbidden & set(frame.columns))

print("normal_train features:", X_train_norm_raw.shape)
print("normal_val   features:", X_val_norm_raw.shape)
print("test         features:", X_test_raw.shape, "(mixture)")
print("\ntest class balance:")
display(y_test[LABEL_COLUMN].value_counts().rename({0: "normal", 1: "attack"}).to_frame("count"))

normal_train features: (29600, 42)
normal_val   features: (7400, 42)
test         features: (175341, 42) (mixture)

test class balance:


,count
label,
attack,119341
normal,56000


## 4. Fit preprocessing on `normal_train` ONLY

**Purpose.** The scaler and encoders learn *only* from `normal_train`. This is the
definitional core of one-class novelty detection: "normal" is what the model knows;
everything else (including test normal flows) is scored against it. The three
representations from notebook 02 are all produced.

In [5]:
representations = ["ohe_full", "ohe_topk", "freq"]
pipelines, outputs = {}, {}

for name in representations:
    pipe = FeaturePipeline(categorical_strategy=name).fit(X_train_norm_raw)
    M_tr = pipe.transform(X_train_norm_raw)   # normal_train
    M_va = pipe.transform(X_val_norm_raw)     # normal_val
    M_te = pipe.transform(X_test_raw)         # held-out mixture
    pipelines[name], outputs[name] = pipe, (M_tr, M_va, M_te)
    print(f"{name:9s}: normal_train {M_tr.shape} | normal_val {M_va.shape} "
          f"| test {M_te.shape}")

ohe_full : normal_train (29600, 58) | normal_val (7400, 58) | test (175341, 58)


ohe_topk : normal_train (29600, 58) | normal_val (7400, 58) | test (175341, 58)


freq     : normal_train (29600, 42) | normal_val (7400, 42) | test (175341, 42)


## 5. Integrity, alignment and leakage checks

**Purpose.** Verify: `normal_val` is normal-only; all three matrices share identical
columns and compatible dimensions; everything is finite; no label/identifier name is
present.

In [6]:
audit_rows = []
for name in representations:
    pipe = pipelines[name]
    M_tr, M_va, M_te = outputs[name]
    names = pipe.get_feature_names()

    check_finite(M_tr, f"{name} normal_train")
    check_finite(M_va, f"{name} normal_val")
    check_finite(M_te, f"{name} test")
    assert M_tr.shape[1] == M_va.shape[1] == M_te.shape[1] == len(names)
    assert not (forbidden & set(names))

    # normal_val contains normal traffic only
    assert (y_normal_val[LABEL_COLUMN] == 0).all()
    assert (y_normal_train[LABEL_COLUMN] == 0).all()

    audit_rows.append({
        "representation": name,
        "n_features": M_tr.shape[1],
        "normal_train": M_tr.shape[0],
        "normal_val": M_va.shape[0],
        "test": M_te.shape[0],
        "dtype": str(M_tr.dtype),
        "all_finite": bool(np.isfinite(M_tr).all() and np.isfinite(M_va).all()
                           and np.isfinite(M_te).all()),
        "columns_aligned": True,
        "val_is_normal_only": True,
    })
audit = pd.DataFrame(audit_rows).set_index("representation")
display(audit)
audit.to_csv(METRICS_DIR / "04_chapter_b_audit.csv")
print("Saved ->", METRICS_DIR / "04_chapter_b_audit.csv")

,n_features,normal_train,normal_val,test,dtype,all_finite,columns_aligned,val_is_normal_only
representation,,,,,,,,
ohe_full,58,29600,7400,175341,float32,True,True,True
ohe_topk,58,29600,7400,175341,float32,True,True,True
freq,42,29600,7400,175341,float32,True,True,True


Saved -> /home/aminelby/Network-Traffic-Clustering/results/metrics/04_chapter_b_audit.csv


**Why the Chapter-B width (58) differs from the Chapter-A width (190).** The
Chapter-A pipeline is fit on the *whole* training split, so its categorical
vocabulary holds all 131 `proto` values. The Chapter-B pipeline is deliberately
fit on `normal_train` **only**, so its vocabulary holds just the handful of
protocols/services/states that normal traffic actually uses. Consequently the
normal model has no columns for attack-only categories — those flows are handled
at transform time (`"other"` for top-K, all-zero for full one-hot). This is not a
bug: it is exactly the "model of normal" construction the proposal requires, and it
is why `ohe_full` and `ohe_topk` happen to coincide in width here (few normal
categories survive either encoding). Within Chapter B, all three matrices share
identical, aligned columns.

In [7]:
# Explicit leakage assertions reproduced as standalone checks.
assert (y_normal_train[LABEL_COLUMN] == 0).all(), "attack label in normal_train!"
assert (y_normal_val[LABEL_COLUMN] == 0).all(), "attack label in normal_val!"
# No `id` (the true row key) overlap between the two normal splits.
assert set(normal_train_df["id"]).isdisjoint(set(normal_val_df["id"]))
assert len(set(normal_train_df["id"]) | set(normal_val_df["id"])) == len(normal_df)

# Confirm the scaler was fit on normal_train only: StandardScaler centres its
# training data, so the scaled numeric block of normal_train must be ~zero-mean.
# It would NOT be zero-mean if the scaler had been fit on the full mixed set.
pipe = pipelines["ohe_topk"]
n_num = len(pipe.numeric_features)
train_numeric_scaled = pipe.transform(X_train_norm_raw)[:, :n_num]
print("max |column mean| of scaled numeric block on normal_train:",
      float(np.abs(train_numeric_scaled.mean(axis=0)).max()))
assert np.allclose(train_numeric_scaled.mean(axis=0), 0.0, atol=1e-4)
print("[OK] Scaler is centred on normal_train only; leakage checks passed.")

max |column mean| of scaled numeric block on normal_train: 3.3188509860337945e-06
[OK] Scaler is centred on normal_train only; leakage checks passed.


## 6. Save outputs

**Purpose.** Persist the three Chapter-B feature matrices per representation plus
the separately-held reference labels and the split definition, so Phase 7/8 can run
without re-fitting.

In [8]:
for name in representations:
    M_tr, M_va, M_te = outputs[name]
    pipe = pipelines[name]
    np.savez_compressed(CHAPTER_B_DIR / f"X_normal_train_{name}.npz", X=M_tr)
    np.savez_compressed(CHAPTER_B_DIR / f"X_normal_val_{name}.npz", X=M_va)
    np.savez_compressed(CHAPTER_B_DIR / f"X_test_{name}.npz", X=M_te)
    pd.Series(pipe.get_feature_names(), name="feature_name").to_csv(
        CHAPTER_B_DIR / f"feature_names_{name}.csv", index=False)
    joblib.dump(pipe, CHAPTER_B_DIR / f"pipeline_{name}.joblib")
    del outputs[name]

# Reference labels, kept strictly separate from the matrices.
y_normal_train.to_csv(CHAPTER_B_DIR / "y_normal_train.csv", index=False)
y_normal_val.to_csv(CHAPTER_B_DIR / "y_normal_val.csv", index=False)
y_test.to_csv(CHAPTER_B_DIR / "y_test.csv", index=False)

# Split definition (reproducible record of which flow went where).
split_record = pd.concat([
    pd.DataFrame({"id": normal_train_df["id"], "split": "normal_train"}),
    pd.DataFrame({"id": normal_val_df["id"], "split": "normal_val"}),
], ignore_index=True)
split_record.to_csv(CHAPTER_B_DIR / "normal_split.csv", index=False)

print("Saved to", CHAPTER_B_DIR)
for p in sorted(CHAPTER_B_DIR.glob("*")):
    print(f"  {p.name:34s} {p.stat().st_size/1024:8.1f} KB")

Saved to /home/aminelby/Network-Traffic-Clustering/data/processed/chapter_b
  X_normal_train_freq.npz              1771.4 KB
  X_normal_train_ohe_full.npz          1852.6 KB
  X_normal_train_ohe_topk.npz          1852.5 KB
  X_normal_val_freq.npz                 440.8 KB
  X_normal_val_ohe_full.npz             461.3 KB
  X_normal_val_ohe_topk.npz             461.3 KB
  X_test_freq.npz                      6687.0 KB
  X_test_ohe_full.npz                  6954.5 KB
  X_test_ohe_topk.npz                  6955.0 KB
  feature_names_freq.csv                  0.4 KB
  feature_names_ohe_full.csv              0.6 KB
  feature_names_ohe_topk.csv              0.6 KB
  normal_split.csv                      671.5 KB
  pipeline_freq.joblib                    5.6 KB
  pipeline_ohe_full.joblib                3.6 KB
  pipeline_ohe_topk.joblib                4.5 KB
  y_normal_train.csv                    260.2 KB
  y_normal_val.csv                       65.1 KB
  y_test.csv                           171

## 7. Reload-and-verify (round trip)

**Purpose.** Confirm the saved normal-test matrix is exactly reproducible from the
saved pipeline and the raw test frame.

In [9]:
name = "ohe_topk"
pipe_reloaded = joblib.load(CHAPTER_B_DIR / f"pipeline_{name}.joblib")
M_te_reloaded = np.load(CHAPTER_B_DIR / f"X_test_{name}.npz")["X"]
M_te_recomputed = pipe_reloaded.transform(X_test_raw)
assert np.allclose(M_te_reloaded, M_te_recomputed)
assert np.isfinite(M_te_reloaded).all()
print(f"[OK] Reloaded '{name}': test shape {M_te_reloaded.shape}, finite, reproducible.")

[OK] Reloaded 'ohe_topk': test shape (175341, 58), finite, reproducible.


## 8. Summary

Chapter B data preparation is complete and verified:

* `normal_train` (80%) and `normal_val` (20%) are disjoint, seeded, stratified by
  `service`, and contain **normal traffic only**.
* The test set is the untouched official mixture (normal + attack).
* Preprocessing is fit **exclusively on `normal_train`**; `normal_val` and the test
  set are transformed with the identical fitted transformers.
* All matrices are finite and column-aligned; reference labels are separate.
* **No K-Means, GMM, Logistic Regression or any model was trained.**

Outputs live in `data/processed/chapter_b/` and feed Phase 7 (build the normal
models) and Phase 8 (freeze budget thresholds on `normal_val`, score the test set once).

In [10]:
summary = [
    "# 04 - Chapter B data preparation summary",
    "",
    "Normal definition: label == 0. Preprocessing fitted on normal_train ONLY.",
    "",
    f"- normal_train: {len(normal_train_df)} flows (80%, stratified by service)",
    f"- normal_val:   {len(normal_val_df)} flows (20%, normal only)",
    f"- held-out test: {len(test_df)} flows (mixture, "
    f"{int((test_df[LABEL_COLUMN]==0).sum())} normal / "
    f"{int((test_df[LABEL_COLUMN]==1).sum())} attack)",
    "",
    "| representation | n_features | normal_train | normal_val | test |",
    "|---|---|---|---|---|",
]
for name in representations:
    a = audit.loc[name]
    summary.append(f"| {name} | {int(a['n_features'])} | {int(a['normal_train'])} | "
                   f"{int(a['normal_val'])} | {int(a['test'])} |")
summary += [
    "",
    "- normal_val is normal-only; test untouched during fitting.",
    "- labels saved separately (y_normal_train / y_normal_val / y_test).",
    "- split record: normal_split.csv; fitted pipelines: pipeline_<rep>.joblib.",
    "- all matrices finite and column-aligned; round-trip verified.",
]
(REPORTS_DIR / "04_chapter_b_summary.md").write_text("\n".join(summary))
print("Saved ->", REPORTS_DIR / "04_chapter_b_summary.md")
print("\n=== Notebook 04 complete: Chapter B data prepared and verified. ===")

Saved -> /home/aminelby/Network-Traffic-Clustering/results/reports/04_chapter_b_summary.md

=== Notebook 04 complete: Chapter B data prepared and verified. ===
